# SafeDose AI: Clinical Multi-Task ML Training Suite (Kaggle Cloud Accelerator)

This notebook executes the end-to-end multi-task clinical machine learning training pipeline for **SafeDose AI (AnesthAI)**.

### Recommended Kaggle Settings:
1. **Accelerator**: `GPU T4 x2` or `GPU P100` (In Notebook Settings sidebar -> **Accelerator**)
2. **Internet**: **ON** (Required for installing dependencies and downloading PhysioNet/VitalDB data)
3. **Persistence**: `Files only`

---

## Step 1: Environment & GPU Verification

In [ ]:
import sys
import os
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
cuda_available = torch.cuda.is_available()
print(f"CUDA (GPU) Available: {cuda_available}")
if cuda_available:
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Count: {torch.cuda.device_count()}")
else:
    print("Running on CPU. For faster training, enable GPU under Notebook Settings -> Accelerator -> GPU T4 x2.")

## Step 2: Install Specialized Clinical Dependencies
Installs packages for PhysioNet waveform parsing (`wfdb`), advanced gradient boosting (`catboost`), tabular deep learning (`pytorch-tabnet`), class balancing (`imbalanced-learn`), and model explainability (`shap`).

In [ ]:
%pip install -q wfdb catboost optuna shap imbalanced-learn pytorch-tabnet
print("Dependencies successfully installed!")

## Step 3: Automated Dataset Ingestion & Preparation
Executes `download_datasets.py` to:
- Verify pre-packaged waveforms (MIT-BIH, VitalDB, MIMIC Demo)
- Download or expand records directly from PhysioNet via WFDB if needed
- Auto-detect and symlink any full MIMIC-IV 3.1 databases if mounted in `/kaggle/input/`

In [ ]:
!python download_datasets.py

## Step 4: Execute Full SafeDose Multi-Task Training Pipeline
Trains and benchmarks candidate models across all 16 clinical perioperative tasks:
- **Intraoperative Hemodynamics**: Hypotension, Hypoxia, Bradycardia, Tachycardia
- **Electrophysiology**: MIT-BIH Cardiac Arrhythmia Detection
- **Preoperative Evaluation**: Difficult Airway (Mallampati), ASA Risk, PONV (Apfel)
- **Critical Care & Labs**: Severe Blood Loss, Goal-Directed Fluid Therapy, ICU Need, Mortality
- **Postoperative & Pharmacology**: PACU Delayed Recovery, Drug Safety Matrix, Prototype Baseline

In [ ]:
!python training/train_pipeline.py --batch-size 256 --epochs 6

## Step 5: Verify All 16 Trained Clinical AI Models
Runs real-time clinical inference tests across all 16 tasks to verify precision, calibration, and clinical action guidelines.

In [ ]:
!python test_all_predictions.py

## Step 6: Package Trained Weights & Plots for 1-Click Download
Compresses `models/` (.pkl & .pt files) and `reports/` (training report & XAI plots) into `SafeDose_Trained_Models.zip` in `/kaggle/working/`.

In [ ]:
import shutil
import zipfile
import os

archive_name = 'SafeDose_Trained_Models.zip'
print(f"[*] Packaging models and reports into {archive_name}...")

with zipfile.ZipFile(archive_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk('models'):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, os.path.relpath(fp, '.'))
            
    for root, dirs, files in os.walk('reports'):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, os.path.relpath(fp, '.'))

size_mb = os.path.getsize(archive_name) / (1024 * 1024)
print(f"[+] Package created successfully: {archive_name} ({size_mb:.2f} MB)")
print("\nYou can now download 'SafeDose_Trained_Models.zip' directly from the Kaggle Output file explorer on the right-hand panel!")